# Toxic Comment Classification

Multi-label text classification system to identify toxic comments across 6 categories using NLP and machine learning.

## Problem Statement
Identify toxic online comments to help platforms moderate content. The dataset has severe class imbalance (only 9.6% toxic comments overall, with some categories <1%).

## Dataset
- **Source**: Jigsaw Toxic Comment Classification Challenge (Kaggle)
- **Size**: 159,571 comments
- **Labels**: 6 categories (toxic, severe_toxic, obscene, threat, insult, identity_hate)
- **Challenge**: Highly imbalanced multi-label classification

In [2]:
import warnings
warnings.filterwarnings("ignore")
import pandas as pd
import numpy as np
from sklearn.pipeline import Pipeline
from sklearn.linear_model import LogisticRegression
from sklearn.naive_bayes import MultinomialNB
from xgboost import XGBClassifier
from sklearn.model_selection import train_test_split
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.multiclass import OneVsRestClassifier
from sklearn.metrics import accuracy_score
from sklearn.metrics import multilabel_confusion_matrix, classification_report
from sklearn.metrics import average_precision_score
from sklearn.dummy import DummyClassifier

## 1. Data Loading and Exploration

In [3]:
raw_df = pd.read_csv("data/train.csv")
raw_df.shape

(159571, 8)

## 2. Exploratory Data Analysis

### Dataset Overview

In [4]:
raw_df.info()

<class 'pandas.DataFrame'>
RangeIndex: 159571 entries, 0 to 159570
Data columns (total 8 columns):
 #   Column         Non-Null Count   Dtype
---  ------         --------------   -----
 0   id             159571 non-null  str  
 1   comment_text   159571 non-null  str  
 2   toxic          159571 non-null  int64
 3   severe_toxic   159571 non-null  int64
 4   obscene        159571 non-null  int64
 5   threat         159571 non-null  int64
 6   insult         159571 non-null  int64
 7   identity_hate  159571 non-null  int64
dtypes: int64(6), str(2)
memory usage: 9.7 MB


In [5]:
raw_df.describe()

,toxic,severe_toxic,obscene,threat,insult,identity_hate
count,159571.000000,159571.000000,159571.000000,159571.000000,159571.000000,159571.000000
mean,0.095844,0.009996,0.052948,0.002996,0.049364,0.008805
std,0.294379,0.099477,0.223931,0.054650,0.216627,0.093420
min,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000
25%,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000
50%,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000
75%,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000
max,1.000000,1.000000,1.000000,1.000000,1.000000,1.000000


In [6]:
raw_df.isna().sum()

id               0
comment_text     0
toxic            0
severe_toxic     0
obscene          0
threat           0
insult           0
identity_hate    0
dtype: int64

In [7]:
df = raw_df.copy()

In [8]:
#df.drop(columns = ['id'], inplace = True)

### Class Distribution Analysis
Understanding imbalance is critical for toxic comment detection.

In [9]:
print(df[['toxic', 'severe_toxic', 'obscene', 'threat', 'insult', 'identity_hate']].sum())
print(df[['toxic', 'severe_toxic', 'obscene', 'threat', 'insult', 'identity_hate']].mean() * 100)

toxic            15294
severe_toxic      1595
obscene           8449
threat             478
insult            7877
identity_hate     1405
dtype: int64
toxic            9.584448
severe_toxic     0.999555
obscene          5.294822
threat           0.299553
insult           4.936361
identity_hate    0.880486
dtype: float64


## 3. Data Preparation

### Train-Test Split


In [10]:
target_cols = ['toxic', 'severe_toxic', 'obscene', 'threat', 'insult', 'identity_hate']
input_col = 'comment_text'

X = df[input_col]
y = df[target_cols]

X_train, X_test, y_train, y_test = train_test_split(X, y, random_state=42)

print("Shape of training input columns (X_train):  ",X_train.shape)
print("Shape of test input columns (X_test):       ",X_test.shape)
print("Shape of training target columns (y_train): ",y_train.shape)
print("Shape of test target columns (y_test):      ",y_test.shape)

Shape of training input columns (X_train):   (119678,)
Shape of test input columns (X_test):        (39893,)
Shape of training target columns (y_train):  (119678, 6)
Shape of test target columns (y_test):       (39893, 6)


In [11]:
print(type(y_train))
print(y_train.shape)


<class 'pandas.DataFrame'>
(119678, 6)


## 4. Model Training & Evaluation
### Model 1: Logistic Regression (Baseline)

In [12]:
pipe_lr = Pipeline([('tfidf_lr', TfidfVectorizer(
        ngram_range=(1,2),
        max_features=50000,
        sublinear_tf=True)),
                    ('model_lr', OneVsRestClassifier(LogisticRegression(solver='liblinear',
                                                                         class_weight='balanced',
                                                                           max_iter=1000,
                                                                             random_state=42)))])

pipe_lr.fit(X_train, y_train)

,"steps steps: list of tuplesList of (name of step, estimator) tuples that are to be chained insequential order. To be compatible with the scikit-learn API, all stepsmust define `fit`. All non-last steps must also define `transform`. See:ref:`Combining Estimators ` for more details.","[('tfidf_lr', ...), ('model_lr', ...)]"
,"transform_input transform_input: list of str, default=NoneThe names of the :term:`metadata` parameters that should be transformed by thepipeline before passing it to the step consuming it.This enables transforming some input arguments to ``fit`` (other than ``X``)to be transformed by the steps of the pipeline up to the step which requiresthem. Requirement is defined via :ref:`metadata routing `.For instance, this can be used to pass a validation set through the pipeline.You can only set this if metadata routing is enabled, which youcan enable using ``sklearn.set_config(enable_metadata_routing=True)``... versionadded:: 1.6",None
,"memory memory: str or object with the joblib.Memory interface, default=NoneUsed to cache the fitted transformers of the pipeline. The last stepwill never be cached, even if it is a transformer. By default, nocaching is performed. If a string is given, it is the path to thecaching directory. Enabling caching triggers a clone of the transformersbefore fitting. Therefore, the transformer instance given to thepipeline cannot be inspected directly. Use the attribute ``named_steps``or ``steps`` to inspect estimators within the pipeline. Caching thetransformers is advantageous when fitting is time consuming. See:ref:`sphx_glr_auto_examples_neighbors_plot_caching_nearest_neighbors.py`for an example on how to enable caching.",None
,"verbose verbose: bool, default=FalseIf True, the time elapsed while fitting each step will be printed as itis completed.",False
,"input input: {'filename', 'file', 'content'}, default='content'- If `'filename'`, the sequence passed as an argument to fit is expected to be a list of filenames that need reading to fetch the raw content to analyze.- If `'file'`, the sequence items must have a 'read' method (file-like object) that is called to fetch the bytes in memory.- If `'content'`, the input is expected to be a sequence of items that can be of type string or byte.",'content'
,"encoding encoding: str, default='utf-8'If bytes or files are given to analyze, this encoding is used todecode.",'utf-8'
,"decode_error decode_error: {'strict', 'ignore', 'replace'}, default='strict'Instruction on what to do if a byte sequence is given to analyze thatcontains characters not of the given `encoding`. By default, it is'strict', meaning that a UnicodeDecodeError will be raised. Othervalues are 'ignore' and 'replace'.",'strict'
,"strip_accents strip_accents: {'ascii', 'unicode'} or callable, default=NoneRemove accents and perform other character normalizationduring the preprocessing step.'ascii' is a fast method that only works on characters that havea direct ASCII mapping.'unicode' is a slightly slower method that works on any characters.None (default) means no character normalization is performed.Both 'ascii' and 'unicode' use NFKD normalization from:func:`unicodedata.normalize`.",None
,"lowercase lowercase: bool, default=TrueConvert all characters to lowercase before tokenizing.",True
,"preprocessor preprocessor: callable, default=NoneOverride the preprocessing (string transformation) stage whilepreserving the tokenizing and n-grams generation steps.Only applies if ``analyzer`` is not callable.",None
,"tokenizer tokenizer: callable, default=NoneOverride the string tokenization step while preserving thepreprocessing and n-grams generation steps.Only applies if ``analyzer == 'word'``.",None


In [13]:
y_test_pred_lr = pipe_lr.predict(X_test)
y_test_pred_lr

array([[1, 0, 0, 0, 0, 0],
       [0, 0, 0, 0, 0, 0],
       [0, 0, 0, 0, 0, 0],
       ...,
       [0, 0, 0, 0, 0, 0],
       [0, 0, 0, 0, 0, 0],
       [0, 0, 0, 0, 0, 0]], shape=(39893, 6))

In [14]:
multilabel_confusion_matrix(y_test, y_test_pred_lr)

array([[[34326,  1752],
        [  547,  3268]],

       [[38711,   776],
        [   75,   331]],

       [[36945,   805],
        [  266,  1877]],

       [[39570,   218],
        [   28,    77]],

       [[36604,  1278],
        [  267,  1744]],

       [[38782,   754],
        [   99,   258]]])

In [15]:
print(classification_report(y_test, y_test_pred_lr))

              precision    recall  f1-score   support

           0       0.65      0.86      0.74      3815
           1       0.30      0.82      0.44       406
           2       0.70      0.88      0.78      2143
           3       0.26      0.73      0.39       105
           4       0.58      0.87      0.69      2011
           5       0.25      0.72      0.38       357

   micro avg       0.58      0.85      0.69      8837
   macro avg       0.46      0.81      0.57      8837
weighted avg       0.61      0.85      0.71      8837
 samples avg       0.06      0.08      0.07      8837



In [16]:
y_test_proba_lr = pipe_lr.predict_proba(X_test)

pr_auc = average_precision_score(y_test, y_test_proba_lr, average='macro')
print("LR Macro PR-AUC: ", pr_auc)

LR Macro PR-AUC:  0.6455228868652892


## Model Interpretation 

In [17]:
labels = ['toxic', 'severe_toxic', 'obscene', 'threat', 'insult', 'identity_hate']

tfidf = pipe_lr.named_steps['tfidf_lr']
ovr = pipe_lr.named_steps['model_lr']

feature_names = tfidf.get_feature_names_out()

top_features = {}

for label, clf in zip(labels, ovr.estimators_):
    coefs = clf.coef_[0]
    top_idx = np.argsort(coefs)[-20:][::-1]
    top_features[label] = pd.DataFrame({
        'feature': feature_names[top_idx],
        'weight': coefs[top_idx]
    })

for label, df_feat in top_features.items():
    print(f"\nTop features for {label}")
    display(df_feat)


Top features for toxic


,feature,weight
0,fuck,18.623039
1,fucking,16.726426
2,stupid,16.510176
3,shit,15.480581
4,idiot,13.558959
5,ass,11.171605
6,crap,10.899299
7,bullshit,10.722366
8,asshole,10.326224
9,suck,10.229090



Top features for severe_toxic


,feature,weight
0,fucking,13.254760
1,fuck,11.921760
2,shit,7.717212
3,asshole,7.160868
4,bitch,6.956124
5,dick,6.802012
6,fucker,6.716228
7,motherfucker,6.679734
8,fuckin,6.547802
9,die,6.503044



Top features for obscene


...

,feature,weight
0,die,16.970980
1,kill,16.161330
2,will,11.813648
3,death,8.946958
4,dead,8.763839
5,you,7.538656
6,shoot,7.480897
7,destroy,7.184914
8,rape,7.136359
9,burn,6.919008



Top features for insult


,feature,weight
0,idiot,15.517396
1,stupid,14.374737
2,asshole,13.127819
3,fucking,12.844890
4,fuck,12.481431
5,bitch,12.296270
6,you,10.784495
7,faggot,10.304163
8,moron,10.273043
9,ass,9.838362



Top features for identity_hate


,feature,weight
0,nigger,16.679331
1,gay,15.881694
2,faggot,12.666162
3,homosexual,11.984003
4,jew,10.577120
5,niggers,10.114570
6,nigga,10.112805
7,nazi,9.579247
8,homo,8.654215
9,jews,8.509721


### Model 2: Naive Bayes

In [18]:
pipe_nb = Pipeline([('tfidf_nb', TfidfVectorizer(ngram_range=(1,2),max_features=50000,sublinear_tf=True)),
                    ('model_nb', OneVsRestClassifier(MultinomialNB(alpha=0.1)))])
pipe_nb.fit(X_train, y_train)

,"steps steps: list of tuplesList of (name of step, estimator) tuples that are to be chained insequential order. To be compatible with the scikit-learn API, all stepsmust define `fit`. All non-last steps must also define `transform`. See:ref:`Combining Estimators ` for more details.","[('tfidf_nb', ...), ('model_nb', ...)]"
,"transform_input transform_input: list of str, default=NoneThe names of the :term:`metadata` parameters that should be transformed by thepipeline before passing it to the step consuming it.This enables transforming some input arguments to ``fit`` (other than ``X``)to be transformed by the steps of the pipeline up to the step which requiresthem. Requirement is defined via :ref:`metadata routing `.For instance, this can be used to pass a validation set through the pipeline.You can only set this if metadata routing is enabled, which youcan enable using ``sklearn.set_config(enable_metadata_routing=True)``... versionadded:: 1.6",None
,"memory memory: str or object with the joblib.Memory interface, default=NoneUsed to cache the fitted transformers of the pipeline. The last stepwill never be cached, even if it is a transformer. By default, nocaching is performed. If a string is given, it is the path to thecaching directory. Enabling caching triggers a clone of the transformersbefore fitting. Therefore, the transformer instance given to thepipeline cannot be inspected directly. Use the attribute ``named_steps``or ``steps`` to inspect estimators within the pipeline. Caching thetransformers is advantageous when fitting is time consuming. See:ref:`sphx_glr_auto_examples_neighbors_plot_caching_nearest_neighbors.py`for an example on how to enable caching.",None
,"verbose verbose: bool, default=FalseIf True, the time elapsed while fitting each step will be printed as itis completed.",False
,"input input: {'filename', 'file', 'content'}, default='content'- If `'filename'`, the sequence passed as an argument to fit is expected to be a list of filenames that need reading to fetch the raw content to analyze.- If `'file'`, the sequence items must have a 'read' method (file-like object) that is called to fetch the bytes in memory.- If `'content'`, the input is expected to be a sequence of items that can be of type string or byte.",'content'
,"encoding encoding: str, default='utf-8'If bytes or files are given to analyze, this encoding is used todecode.",'utf-8'
,"decode_error decode_error: {'strict', 'ignore', 'replace'}, default='strict'Instruction on what to do if a byte sequence is given to analyze thatcontains characters not of the given `encoding`. By default, it is'strict', meaning that a UnicodeDecodeError will be raised. Othervalues are 'ignore' and 'replace'.",'strict'
,"strip_accents strip_accents: {'ascii', 'unicode'} or callable, default=NoneRemove accents and perform other character normalizationduring the preprocessing step.'ascii' is a fast method that only works on characters that havea direct ASCII mapping.'unicode' is a slightly slower method that works on any characters.None (default) means no character normalization is performed.Both 'ascii' and 'unicode' use NFKD normalization from:func:`unicodedata.normalize`.",None
,"lowercase lowercase: bool, default=TrueConvert all characters to lowercase before tokenizing.",True
,"preprocessor preprocessor: callable, default=NoneOverride the preprocessing (string transformation) stage whilepreserving the tokenizing and n-grams generation steps.Only applies if ``analyzer`` is not callable.",None
,"tokenizer tokenizer: callable, default=NoneOverride the string tokenization step while preserving thepreprocessing and n-grams generation steps.Only applies if ``analyzer == 'word'``.",None


In [19]:
y_test_pred_nb = pipe_nb.predict(X_test)

In [20]:
multilabel_confusion_matrix(y_test, y_test_pred_nb)

array([[[35543,   535],
        [ 1396,  2419]],

       [[39278,   209],
        [  211,   195]],

       [[37373,   377],
        [  777,  1366]],

       [[39765,    23],
        [   91,    14]],

       [[37398,   484],
        [  815,  1196]],

       [[39385,   151],
        [  291,    66]]])

In [21]:
print(classification_report(y_test, y_test_pred_nb))

              precision    recall  f1-score   support

           0       0.82      0.63      0.71      3815
           1       0.48      0.48      0.48       406
           2       0.78      0.64      0.70      2143
           3       0.38      0.13      0.20       105
           4       0.71      0.59      0.65      2011
           5       0.30      0.18      0.23       357

   micro avg       0.75      0.59      0.66      8837
   macro avg       0.58      0.44      0.50      8837
weighted avg       0.74      0.59      0.66      8837
 samples avg       0.05      0.05      0.05      8837



In [22]:
y_test_proba_nb = pipe_nb.predict_proba(X_test)

pr_auc = average_precision_score(y_test, y_test_proba_nb, average='macro')
print("NB Macro PR-AUC: ", pr_auc)


NB Macro PR-AUC:  0.5216166119294597


### Model 3: XGBoost

In [23]:
pipe_xgb = Pipeline([('tfidf_xgb', TfidfVectorizer(ngram_range=(1,2), max_features=50000, sublinear_tf=True)),
                     ('model_xgb', XGBClassifier(scale_pos_weight = 13, eval_metric='aucpr', random_state = 42))])
pipe_xgb.fit(X_train, y_train)


,"steps steps: list of tuplesList of (name of step, estimator) tuples that are to be chained insequential order. To be compatible with the scikit-learn API, all stepsmust define `fit`. All non-last steps must also define `transform`. See:ref:`Combining Estimators ` for more details.","[('tfidf_xgb', ...), ('model_xgb', ...)]"
,"transform_input transform_input: list of str, default=NoneThe names of the :term:`metadata` parameters that should be transformed by thepipeline before passing it to the step consuming it.This enables transforming some input arguments to ``fit`` (other than ``X``)to be transformed by the steps of the pipeline up to the step which requiresthem. Requirement is defined via :ref:`metadata routing `.For instance, this can be used to pass a validation set through the pipeline.You can only set this if metadata routing is enabled, which youcan enable using ``sklearn.set_config(enable_metadata_routing=True)``... versionadded:: 1.6",None
,"memory memory: str or object with the joblib.Memory interface, default=NoneUsed to cache the fitted transformers of the pipeline. The last stepwill never be cached, even if it is a transformer. By default, nocaching is performed. If a string is given, it is the path to thecaching directory. Enabling caching triggers a clone of the transformersbefore fitting. Therefore, the transformer instance given to thepipeline cannot be inspected directly. Use the attribute ``named_steps``or ``steps`` to inspect estimators within the pipeline. Caching thetransformers is advantageous when fitting is time consuming. See:ref:`sphx_glr_auto_examples_neighbors_plot_caching_nearest_neighbors.py`for an example on how to enable caching.",None
,"verbose verbose: bool, default=FalseIf True, the time elapsed while fitting each step will be printed as itis completed.",False
,"input input: {'filename', 'file', 'content'}, default='content'- If `'filename'`, the sequence passed as an argument to fit is expected to be a list of filenames that need reading to fetch the raw content to analyze.- If `'file'`, the sequence items must have a 'read' method (file-like object) that is called to fetch the bytes in memory.- If `'content'`, the input is expected to be a sequence of items that can be of type string or byte.",'content'
,"encoding encoding: str, default='utf-8'If bytes or files are given to analyze, this encoding is used todecode.",'utf-8'
,"decode_error decode_error: {'strict', 'ignore', 'replace'}, default='strict'Instruction on what to do if a byte sequence is given to analyze thatcontains characters not of the given `encoding`. By default, it is'strict', meaning that a UnicodeDecodeError will be raised. Othervalues are 'ignore' and 'replace'.",'strict'
,"strip_accents strip_accents: {'ascii', 'unicode'} or callable, default=NoneRemove accents and perform other character normalizationduring the preprocessing step.'ascii' is a fast method that only works on characters that havea direct ASCII mapping.'unicode' is a slightly slower method that works on any characters.None (default) means no character normalization is performed.Both 'ascii' and 'unicode' use NFKD normalization from:func:`unicodedata.normalize`.",None
,"lowercase lowercase: bool, default=TrueConvert all characters to lowercase before tokenizing.",True
,"preprocessor preprocessor: callable, default=NoneOverride the preprocessing (string transformation) stage whilepreserving the tokenizing and n-grams generation steps.Only applies if ``analyzer`` is not callable.",None
,"tokenizer tokenizer: callable, default=NoneOverride the string tokenization step while preserving thepreprocessing and n-grams generation steps.Only applies if ``analyzer == 'word'``.",None


In [24]:
y_test_pred_xgb = pipe_xgb.predict(X_test)

In [25]:
multilabel_confusion_matrix(y_test, y_test_pred_xgb)

array([[[32683,  3395],
        [  556,  3259]],

       [[39099,   388],
        [  173,   233]],

       [[37119,   631],
        [  325,  1818]],

       [[39734,    54],
        [   65,    40]],

       [[36643,  1239],
        [  391,  1620]],

       [[39321,   215],
        [  194,   163]]])

In [26]:
print(classification_report(y_test, y_test_pred_xgb)) 

              precision    recall  f1-score   support

           0       0.49      0.85      0.62      3815
           1       0.38      0.57      0.45       406
           2       0.74      0.85      0.79      2143
           3       0.43      0.38      0.40       105
           4       0.57      0.81      0.67      2011
           5       0.43      0.46      0.44       357

   micro avg       0.55      0.81      0.65      8837
   macro avg       0.51      0.65      0.56      8837
weighted avg       0.56      0.81      0.66      8837
 samples avg       0.07      0.08      0.07      8837



In [27]:
y_test_proba_xgb = pipe_xgb.predict_proba(X_test)

pr_auc = average_precision_score(y_test, y_test_proba_xgb, average='macro')
print("XGB Macro PR-AUC: ", pr_auc)


XGB Macro PR-AUC:  0.6016906938668789


### Baseline Comparison: Dummy Classifier

In [28]:
pipe_dummy = Pipeline([
    ('tfidf',      TfidfVectorizer()),
    ('model_dummy', OneVsRestClassifier(DummyClassifier(strategy='most_frequent')))
])

pipe_dummy.fit(X_train, y_train)

y_test_pred_dummy  = pipe_dummy.predict(X_test)
y_test_proba_dummy = pipe_dummy.predict_proba(X_test)

print(classification_report(y_test, y_test_pred_dummy))
print("Dummy Macro PR-AUC: ", average_precision_score(y_test, y_test_proba_dummy, average='macro'))


              precision    recall  f1-score   support

           0       0.00      0.00      0.00      3815
           1       0.00      0.00      0.00       406
           2       0.00      0.00      0.00      2143
           3       0.00      0.00      0.00       105
           4       0.00      0.00      0.00      2011
           5       0.00      0.00      0.00       357

   micro avg       0.00      0.00      0.00      8837
   macro avg       0.00      0.00      0.00      8837
weighted avg       0.00      0.00      0.00      8837
 samples avg       0.00      0.00      0.00      8837

Dummy Macro PR-AUC:  0.036919593245264413


## 5. Results Comparison

| Model | Macro F1 | Micro F1 | Macro PR-AUC | Key Insight |
|-------|----------|----------|--------------|-------------|
| Dummy (Baseline) | 0.00 | 0.00 | 0.037 | Predicts majority class only (fails to detect toxic labels) |
| Naive Bayes | 0.50 | 0.66 | 0.522 | More conservative, lower recall on rare labels |
| XGBoost | 0.56 | 0.65 | 0.602 | Balanced performance across labels |
| **Logistic Regression** | **0.57** | **0.69** | **0.646** | **Best overall – strongest PR-AUC & stable across labels** |

## 6. Error Analysis

Performed qualitative analysis on false positives and false negatives across rare toxic categories such as `threat` and `identity_hate`.

Observed challenges:
- Context-dependent toxicity
- Sarcasm and implicit insults
- Label overlap between `toxic`, `insult`, and `obscene`
- Rare-category instability due to extreme class imbalance

This analysis helped compare model robustness beyond aggregate metrics and provided deeper insight into model behavior on difficult multi-label text cases.

In [29]:
labels = ['toxic', 'severe_toxic', 'obscene', 'threat', 'insult', 'identity_hate']

X_test_reset = X_test.reset_index(drop=True)
y_test_reset = y_test.reset_index(drop=True)

y_pred_best = pipe_lr.predict(X_test)
y_pred_best_df = pd.DataFrame(y_pred_best, columns=labels).reset_index(drop=True)

In [30]:
def show_label_errors(label_name, n=10):
    true_vals = y_test_reset[label_name].reset_index(drop=True)
    pred_vals = y_pred_best_df[label_name].reset_index(drop=True)

    fp_idx = (true_vals == 0) & (pred_vals == 1)
    fn_idx = (true_vals == 1) & (pred_vals == 0)

    false_positives = pd.DataFrame({
        'comment_text': X_test_reset[fp_idx],
        'true_label': true_vals[fp_idx],
        'pred_label': pred_vals[fp_idx]
    }).head(n)

    false_negatives = pd.DataFrame({
        'comment_text': X_test_reset[fn_idx],
        'true_label': true_vals[fn_idx],
        'pred_label': pred_vals[fn_idx]
    }).head(n)

    return false_positives, false_negatives

In [31]:
fp_threat, fn_threat = show_label_errors('threat', n=5)
fp_identity, fn_identity = show_label_errors('identity_hate', n=5)

print("False Positives - threat")
display(fp_threat)

print("False Negatives - threat")
display(fn_threat)

print("False Positives - identity_hate")
display(fp_identity)

print("False Negatives - identity_hate")
display(fn_identity)

False Positives - threat


,comment_text,true_label,pred_label
226,you fucking piece of shit,0,1
394,LIVE FREE OR DIE TRYING!,0,1
1001,"Fuck off and rape your kids, Fenian bogtrottin...",0,1
1070,"<-WHAT A FAGGOT, I'M GLAD HE'S DEAD!!!",0,1
1107,Fuck off you stupid motherfucker you! Jschnur ...,0,1


False Negatives - threat


,comment_text,true_label,pred_label
373,"Honesty, where the fuck is the Adeputs Mechani...",1,0
1591,Bigger Bang Tour Memorable Moments \n\nWhy did...,1,0
3436,"Dear Snigbrook \n\nYour days are numbered, the...",1,0
7072,hello \n\nI would like to draw your attention ...,1,0
7306,Ahh shut the fuck up you douchebag sand nigger...,1,0


False Positives - identity_hate


,comment_text,true_label,pred_label
26,Renzoy16 ! Shut The Fuck Up,0,1
90,Do not help the jew Schnider. Final warning.,0,1
103,"and you are a motherfucking asshole,suck your ...",0,1
167,"Listen Up! \n\nGo Fuck Yourself, Asswipe!",0,1
217,Reverting constructive edits \n\nWhy are you r...,0,1


False Negatives - identity_hate


,comment_text,true_label,pred_label
232,Why are you STILL harassing me? Do you love B...,1,0
1509,To whoever said most people in arab are homose...,1,0
1510,terrorist \n\nYou seem like a terrorist sir. A...,1,0
1900,"""\nDavid Kellermann\nDID YOU TAG MY PAGE CUNT?...",1,0
2563,REDIRECT Talk:Bolivian chinchilla rat,1,0


### Observations

Qualitative inspection of a sample of false positives and false negatives showed that the model handled many comments with explicit toxic wording more reliably than subtle, implicit, or context-dependent cases.

Common error patterns included:
- Overlap between `toxic`, `insult`, and `obscene`, where abusive language was predicted as a rarer label such as `threat`.
- Ambiguous wording that required conversational context beyond the isolated comment text.
- Missed rare-label examples in `threat` and `identity_hate`, likely due to severe class imbalance.
- Difficulty with indirect or implicit harmful language that was not expressed through obvious keywords.

These findings show that aggregate metrics alone do not fully capture model limitations, especially for rare and semantically nuanced labels in multi-label toxic comment classification.

## LLM-Assisted Toxicity Analysis

To better understand ambiguous and context-dependent toxic comments, I used an LLM to review a small sample of difficult cases from the error analysis section. The goal was not to replace the classifier, but to compare classical TF-IDF predictions with context-aware language interpretation.

This experiment focused on comments that were misclassified in rare or nuanced categories such as `threat` and `identity_hate`, where sarcasm, indirect hostility, and contextual language can make classification difficult.

The LLM was asked to return:
- predicted toxicity labels,
- whether the toxicity was explicit or implicit,
- whether conversational context was required,
- a short explanation of the judgment.

This helped highlight where lexical models performed well and where contextual reasoning added value.

In [32]:
labels = ['toxic', 'severe_toxic', 'obscene', 'threat', 'insult', 'identity_hate']

fp_threat, fn_threat = show_label_errors('threat', n=5)
fp_identity, fn_identity = show_label_errors('identity_hate', n=5)

llm_cases = pd.concat([
    fp_threat.assign(error_type='false_positive', focus_label='threat'),
    fn_threat.assign(error_type='false_negative', focus_label='threat'),
    fp_identity.assign(error_type='false_positive', focus_label='identity_hate'),
    fn_identity.assign(error_type='false_negative', focus_label='identity_hate')
], ignore_index=True)

llm_cases = llm_cases.rename(columns={
    'comment_text': 'comment_text',
    'true_label': 'true_label',
    'pred_label': 'pred_label'
})

llm_cases = llm_cases[['focus_label', 'error_type', 'comment_text', 'true_label', 'pred_label']]
llm_cases

,focus_label,error_type,comment_text,true_label,pred_label
0,threat,false_positive,you fucking piece of shit,0,1
1,threat,false_positive,LIVE FREE OR DIE TRYING!,0,1
2,threat,false_positive,"Fuck off and rape your kids, Fenian bogtrottin...",0,1
3,threat,false_positive,"<-WHAT A FAGGOT, I'M GLAD HE'S DEAD!!!",0,1
4,threat,false_positive,Fuck off you stupid motherfucker you! Jschnur ...,0,1
5,threat,false_negative,"Honesty, where the fuck is the Adeputs Mechani...",1,0
6,threat,false_negative,Bigger Bang Tour Memorable Moments \n\nWhy did...,1,0
7,threat,false_negative,"Dear Snigbrook \n\nYour days are numbered, the...",1,0
8,threat,false_negative,hello \n\nI would like to draw your attention ...,1,0
9,threat,false_negative,Ahh shut the fuck up you douchebag sand nigger...,1,0


In [33]:
import sys
!{sys.executable} -m pip install groq


[notice] A new release of pip is available: 26.0 -> 26.1.2
[notice] To update, run: pip install --upgrade pip


In [34]:
import os
from getpass import getpass
from groq import Groq

os.environ["GROQ_API_KEY"] = getpass("Paste Groq API key: ")
client = Groq(api_key=os.getenv("GROQ_API_KEY"))

In [35]:
from groq import Groq
client = Groq(api_key=os.getenv("GROQ_API_KEY"))

In [36]:
response = client.chat.completions.create(
    model="llama-3.3-70b-versatile",
    messages=[
        {"role": "user", "content": 'Return JSON only: {"test": true}'}
    ]
)

print(response.choices[0].message.content)

{"test": true}


In [37]:
import json
import time
import pandas as pd
import re

labels = ['toxic', 'severe_toxic', 'obscene', 'threat', 'insult', 'identity_hate']

fp_threat, fn_threat = show_label_errors('threat', n=5)
fp_identity, fn_identity = show_label_errors('identity_hate', n=5)

llm_cases = pd.concat([
    fp_threat.assign(error_type='false_positive', focus_label='threat'),
    fn_threat.assign(error_type='false_negative', focus_label='threat'),
    fp_identity.assign(error_type='false_positive', focus_label='identity_hate'),
    fn_identity.assign(error_type='false_negative', focus_label='identity_hate')
], ignore_index=True)

llm_cases = llm_cases[['focus_label', 'error_type', 'comment_text', 'true_label', 'pred_label']]

def extract_json(text):
    text = text.strip()
    text = re.sub(r"^```(?:json)?\s*", "", text)
    text = re.sub(r"\s*```$", "", text)
    return json.loads(text)

def analyze_toxic_comment_with_llm(comment_text):
    prompt = f"""
Return ONLY valid JSON.

Schema:
{{
  "toxic_labels": ["toxic", "severe_toxic", "obscene", "threat", "insult", "identity_hate"],
  "explicit_or_implicit": "explicit" or "implicit",
  "context_required": true or false,
  "explanation": "one short sentence",
  "ambiguity_note": "one short sentence"
}}

Comment:
\"\"\"{comment_text}\"\"\"
"""
    response = client.chat.completions.create(
        model="llama-3.3-70b-versatile",
        messages=[{"role": "user", "content": prompt}],
        temperature=0,
        max_tokens=180,
        response_format={"type": "json_object"}
    )
    return extract_json(response.choices[0].message.content)

rows = []

for _, row in llm_cases.iterrows():
    try:
        result = analyze_toxic_comment_with_llm(row['comment_text'])
        rows.append({
            'focus_label': row['focus_label'],
            'error_type': row['error_type'],
            'comment_text': row['comment_text'],
            'true_label': row['true_label'],
            'pred_label': row['pred_label'],
            'llm_toxic_labels': result.get('toxic_labels', []),
            'llm_explicit_or_implicit': result.get('explicit_or_implicit'),
            'llm_context_required': result.get('context_required'),
            'llm_explanation': result.get('explanation'),
            'llm_ambiguity_note': result.get('ambiguity_note')
        })
        time.sleep(0.5)
    except Exception as e:
        rows.append({
            'focus_label': row['focus_label'],
            'error_type': row['error_type'],
            'comment_text': row['comment_text'],
            'true_label': row['true_label'],
            'pred_label': row['pred_label'],
            'llm_toxic_labels': None,
            'llm_explicit_or_implicit': None,
            'llm_context_required': None,
            'llm_explanation': f'ERROR: {e}',
            'llm_ambiguity_note': None
        })

llm_analysis_df = pd.DataFrame(rows)
llm_analysis_df

,focus_label,error_type,comment_text,true_label,pred_label,llm_toxic_labels,llm_explicit_or_implicit,llm_context_required,llm_explanation,llm_ambiguity_note
0,threat,false_positive,you fucking piece of shit,0,1,"[toxic, severe_toxic, obscene, insult]",explicit,False,The comment contains strong profanity and a se...,The language used is unequivocally toxic.
1,threat,false_positive,LIVE FREE OR DIE TRYING!,0,1,[severe_toxic],explicit,False,The comment contains aggressive language.,The tone is clearly hostile.
2,threat,false_positive,"Fuck off and rape your kids, Fenian bogtrottin...",0,1,"[toxic, severe_toxic, obscene, threat, insult,...",explicit,False,The comment contains extremely offensive langu...,The tone is clearly aggressive and hostile.
3,threat,false_positive,"<-WHAT A FAGGOT, I'M GLAD HE'S DEAD!!!",0,1,"[toxic, severe_toxic, obscene, insult, identit...",explicit,False,The comment contains a slur and expresses glad...,The language used is direct and hateful.
4,threat,false_positive,Fuck off you stupid motherfucker you! Jschnur ...,0,1,"[toxic, severe_toxic, obscene, insult]",explicit,False,The comment contains strong profanity and insu...,The language used is unequivocally toxic.
5,threat,false_negative,"Honesty, where the fuck is the Adeputs Mechani...",1,0,"[toxic, severe_toxic, obscene, insult]",explicit,False,The comment contains profanity and violent lan...,The tone is clearly aggressive and hostile.
6,threat,false_negative,Bigger Bang Tour Memorable Moments \n\nWhy did...,1,0,"[toxic, obscene, insult]",explicit,False,The comment contains strong profanity and insu...,The language used is clearly aggressive and of...
7,threat,false_negative,"Dear Snigbrook \n\nYour days are numbered, the...",1,0,"[threat, insult, identity_hate]",explicit,False,The comment contains direct threats and hate s...,The language used is overtly aggressive and of...
8,threat,false_negative,hello \n\nI would like to draw your attention ...,1,0,"[toxic, severe_toxic, insult, identity_hate]",explicit,False,The comment contains severe insults and hate s...,The language used is highly offensive and leav...
9,threat,false_negative,Ahh shut the fuck up you douchebag sand nigger...,1,0,"[toxic, severe_toxic, obscene, threat, insult,...",explicit,False,The comment contains hate speech and violent t...,The language used is highly offensive and leav...


In [38]:
summary = pd.concat([
    llm_analysis_df['llm_explicit_or_implicit'].value_counts(dropna=False).rename('explicit_or_implicit'),
    llm_analysis_df['llm_context_required'].value_counts(dropna=False).rename('context_required')
], axis=1).fillna(0).astype(int)

summary

,explicit_or_implicit,context_required
explicit,19,0
implicit,1,0
False,0,19
True,0,1


In [42]:
llm_analysis_df['predicted_focus_label'] = llm_analysis_df['llm_toxic_labels'].apply(
    lambda x: x[0] if isinstance(x, list) and len(x) > 0 else None
)

llm_analysis_df['matches_focus'] = llm_analysis_df.apply(
    lambda r: r['focus_label'] in r['llm_toxic_labels'] if isinstance(r['llm_toxic_labels'], list) else False,
    axis=1
)

llm_analysis_df[['focus_label', 'error_type', 'predicted_focus_label', 'matches_focus', 'llm_explicit_or_implicit', 'llm_context_required']]

,focus_label,error_type,predicted_focus_label,matches_focus,llm_explicit_or_implicit,llm_context_required
0,threat,false_positive,toxic,False,explicit,False
1,threat,false_positive,severe_toxic,False,explicit,False
2,threat,false_positive,toxic,True,explicit,False
3,threat,false_positive,toxic,False,explicit,False
4,threat,false_positive,toxic,False,explicit,False
5,threat,false_negative,toxic,False,explicit,False
6,threat,false_negative,toxic,False,explicit,False
7,threat,false_negative,threat,True,explicit,False
8,threat,false_negative,toxic,False,explicit,False
9,threat,false_negative,toxic,True,explicit,False
